            # Round 5: The Final Stretch

            This notebook documents the final `trader.py` submission for the 50
            new Round 5 products plus the Ignith manual Ashflow Alpha allocation.

            ## Final replay summary

            - Combined deterministic replay: **168,602.5 XIRECS**
            - Replay horizon: official-style timestamps `0..99,900`, matching the
              1,000-tick log bundle rather than the exploratory full-day files.
            - Fill model: book-crossing plus public-repo-style passive fills from
              bot trades that would have interacted with our improved quote.
            - Fill count: 515 crossing fills, 1,244 passive fills, 7,430 total filled units
            - Active algorithmic file: `trader.py`
            - Diagnostics script: `scripts/round5_diagnostics.py`

            | Day | Replay PnL |
|---:|---:|
| 2 | 79,122.0 |
| 3 | 38,754.0 |
| 4 | 50,726.5 |

            ## Top replay contributors

            | Product | Replay PnL |
|---|---:|
| `ROBOT_IRONING` | 34,301.0 |
| `OXYGEN_SHAKE_EVENING_BREATH` | 25,640.0 |
| `MICROCHIP_OVAL` | 13,528.0 |
| `PANEL_1X2` | 10,290.0 |
| `OXYGEN_SHAKE_GARLIC` | 9,321.0 |
| `SLEEP_POD_NYLON` | 8,142.0 |
| `UV_VISOR_ORANGE` | 7,003.5 |
| `SLEEP_POD_SUEDE` | 5,634.0 |
| `SNACKPACK_STRAWBERRY` | 5,617.5 |
| `PANEL_2X4` | 5,124.5 |
| `TRANSLATOR_ECLIPSE_CHARCOAL` | 4,991.5 |
| `OXYGEN_SHAKE_CHOCOLATE` | 4,242.0 |
| `PEBBLES_XS` | 4,227.5 |
| `SNACKPACK_PISTACHIO` | 3,948.5 |
| `SNACKPACK_RASPBERRY` | 3,745.0 |
| `TRANSLATOR_ASTRO_BLACK` | 3,482.0 |

## Public IMC repository lessons used

I reviewed public Prosperity writeups and code repositories before
finalizing the Round 5 shape:

| Source | Applicable lesson |
|---|---|
| [Frankfurt Hedgehogs, Prosperity 3, 2nd globally](https://github.com/TimoDiehm/imc-prosperity-3) | Treat Prosperity as a microstructure game first: identify the fair price, improve inside the spread, and use dashboards/backtests to inspect fills. Their writeup emphasizes WallMid/true-price reasoning, inventory clearing, and bot behavior. |
| [Linear Utility, Prosperity 2, 2nd place](https://github.com/ericcccsliu/imc-prosperity-2) | Build a replay harness, grid search simple parameters, and prefer structural edges over decorative correlations. Their most durable edges came from true-price market making, conversion arbitrage, and spread trades. |
| [jmerle, Prosperity 2, 9th overall](https://github.com/jmerle/imc-prosperity-2) | In Round 5, de-anonymized flow can dominate. Their writeup found named-trader directional signals and also warns that overfit directional products can lose badly. |
| [AlphaBaguette, Prosperity 3](https://github.com/Sylvain-Topeza/imc-prosperity-3) | Combine complementary small edges: adaptive market making, informed flow when available, index/spread logic, and strict position limits. |
| [Prosperity preparation discussion](https://www.reddit.com/r/learnquant/comments/1rvf93p/how_to_actually_compete_and_maybe_win_in_imc/) | The common archetypes are fixed-fair market making, basket/stat-arb spreads, options, location arbitrage, and Round 5 trader-ID flow. |

The Round 5 trade files in this dataset do **not** reveal buyer/seller
IDs; every buyer and seller field is blank. Therefore the prior
"copy the informed trader" trick is not directly available here. The
final algorithm instead applies the reusable parts of those writeups:
fair-price market making, fill-aware parameter selection, and only a
few high-confidence directional overlays.

## Local data findings

The zip contains three historical days: days 2, 3, and 4. Each day has
10,000 timestamps for all 50 products.

Main discoveries:

- A full product-by-product isolation pass found that the strongest
  deployable edge is a mix of passive inside-spread market making
  and simple one-tick regime signals.
- `ROBOT_IRONING`, `MICROCHIP_OVAL`, `PANEL_1X2`, and
  `SLEEP_POD_NYLON` show robust large-move reversion. `OXYGEN_SHAKE_GARLIC`
  shows large-move momentum.
- `OXYGEN_SHAKE_CHOCOLATE` and
  `OXYGEN_SHAKE_EVENING_BREATH` have jump-reversion events large
  enough to justify crossing the spread. These are deliberately gated
  by a 30-XIREC one-tick move threshold.
- Cross-sectional group residual/stat-arb tests were mostly flat or
  negative after paying spread. The script keeps those diagnostics,
  but `trader.py` does not deploy a group residual sleeve.
- Products without robust replay contribution are left idle. Unused
  symbols are better than forced variance.

            ## Product-by-product alpha isolation

            `scripts/round5_product_alpha.py` evaluates every product separately
            across passive maker grids, one-tick return regimes, rolling
            z-score regimes, book imbalance/microprice takers, and per-product
            ridge ML takers. Parameters are selected on two days and scored on
            the held-out day.

            | Product | Family | LOO PnL | Worst fold | Full-sample config |
|---|---|---:|---:|---|
| `ROBOT_IRONING` | signals | 27,986.0 | 0.0 | `return_1 {'threshold': 25.0, 'mode': 'reversion', 'sticky': True}` |
| `MICROCHIP_OVAL` | signals | 13,219.0 | 2,843.0 | `return_1 {'threshold': 30.0, 'mode': 'reversion', 'sticky': True}` |
| `OXYGEN_SHAKE_GARLIC` | signals | 10,489.0 | 449.0 | `return_1 {'threshold': 25.0, 'mode': 'momentum', 'sticky': True}` |
| `PANEL_1X2` | signals | 10,290.0 | 1,382.0 | `return_1 {'threshold': 25.0, 'mode': 'reversion', 'sticky': True}` |
| `OXYGEN_SHAKE_EVENING_BREATH` | signals | 9,266.0 | 0.0 | `return_1 {'threshold': 30.0, 'mode': 'reversion', 'sticky': True}` |
| `SLEEP_POD_NYLON` | signals | 6,994.0 | 1,965.0 | `return_1 {'threshold': 25.0, 'mode': 'reversion', 'sticky': True}` |
| `UV_VISOR_ORANGE` | maker | 6,729.5 | 609.0 | `maker {'edge': 3.0, 'skew': 0.0}` |
| `TRANSLATOR_ECLIPSE_CHARCOAL` | maker | 4,661.0 | 255.5 | `maker {'edge': 2.0, 'skew': 0.0}` |
| `SNACKPACK_STRAWBERRY` | maker | 4,615.5 | 472.0 | `maker {'edge': 4.0, 'skew': 0.0}` |
| `SLEEP_POD_SUEDE` | maker | 4,508.0 | 562.5 | `maker {'edge': 2.0, 'skew': 0.0}` |
| `PEBBLES_XS` | maker | 4,227.5 | 1,253.0 | `maker {'edge': 5.0, 'skew': 1.0}` |
| `SNACKPACK_PISTACHIO` | maker | 3,503.5 | 480.0 | `maker {'edge': 0.5, 'skew': 0.5}` |
| `SLEEP_POD_NYLON` | maker | 3,112.0 | 583.0 | `maker {'edge': 4.0, 'skew': 0.0}` |
| `PANEL_2X4` | maker | 3,008.5 | 215.0 | `maker {'edge': 4.0, 'skew': 0.0}` |

            Category-level residual/stat-arb tests were also evaluated. They did
            not clear the robustness bar, so the final bot keeps the simpler
            per-product signals instead.

            Original submitted bot reported 12,180.5 XIRECS. Replaying the current bot on the same 1,000-tick log gives an estimated 43,043.5 XIRECS under the anonymous-trade passive-fill model.

            ## ML, neural net, and transformer research

            I tested ML as a **gate** around the existing strategy, not as an
            unrestricted replacement. The validation is leave-one-day-out:
            train on two historical days, choose thresholds/quote gates only on
            those training days, then score the untouched held-out day.

            Tested families:

            - Ridge regression with product one-hot features.
            - Histogram gradient boosted trees.
            - Multi-layer perceptron neural network.
            - LightGBM boosted trees.
            - Tiny PyTorch transformer over 16-tick order-book sequences.

            | Model | Tested use | Holdout PnL | Delta vs baseline | Mean target corr | Standalone taker PnL |
|---|---|---:|---:|---:|---:|
| `ridge` | passive quote filter | 61,003.0 | -3,297.0 | 0.0336 | 0.0 |
| `hist_gradient_boosting` | passive quote filter | 63,331.0 | -969.0 | 0.0235 | -85,165.0 |
| `mlp` | passive quote filter | 63,656.5 | -643.5 | 0.0334 | -700.0 |
| `lightgbm` | passive quote filter | 61,874.5 | -2,425.5 | 0.0333 | -115,525.0 |
| `tiny_transformer` | passive quote filter | 46,094.0 | -18,206.0 | 0.0274 | n/a |

            Conclusion: the best advanced model still failed to beat the
            current 64,300 XIRECS baseline on leave-one-day-out replay. The
            predictors show tiny next-tick correlations, but the signal is not
            strong enough to pay spread/queue costs. I therefore did **not**
            add ML logic to `trader.py`; the research harness is saved as
            `scripts/round5_ml_research.py` for further experiments.

In [ ]:
import json
from pathlib import Path

diagnostics = json.loads(Path("logs/round5_diagnostics.json").read_text())
diagnostics["combined_pnl"], diagnostics["fills"]

In [ ]:
import pandas as pd

totals = diagnostics["per_product_totals"]
pd.Series(totals).sort_values(ascending=False).head(20).to_frame("replay_pnl")

In [ ]:
ml_research = json.loads(Path("logs/round5_ml_research.json").read_text())
{
    "baseline_sum": ml_research["baseline_sum"],
    "best_tabular_passive": max(
        (
            (name, result["passive_sum"], result["passive_delta"])
            for name, result in ml_research["tabular_models"].items()
        ),
        key=lambda item: item[1],
    ),
    "transformer_delta": ml_research["transformer"].get("passive_delta"),
}

## Trader implementation

The final `trader.py` is Round 5 only. It does not trade any products
from previous rounds.

Strategy layers:

- **Passive selected makers:** quote one tick better than the best
  displayed bid/ask only when the quote still has positive edge to the
  current book mid after inventory skew.
- **Per-product learned signal takers:** six simple one-tick
  return regimes are stored as constants in `SIGNAL_PARAMS`. These
  are distilled from the product-alpha scan instead of running a
  heavyweight model live.
- **Jump-reversion takers:** cross only after very large one-tick
  moves in the two oxygen products where this paid across replay.
- **No anonymous-flow follower:** buyer/seller IDs in the official log
  are blank except for `SUBMISSION`, and cost-aware flow following was
  negative after spread.
- **Risk controls:** all logic respects the hard 10-unit position
  limit per product, and products without robust evidence are idle.

## Ignith manual strategy

Submit the following Ashflow Alpha manual orders:

| Good | Side | % |
|---|---:|---:|
| Sulfur reactor | Buy | 16% |
| Thermalite core | Buy | 14% |
| Lava cake | Sell | 13% |
| Pyroflex cells | Sell | 11% |
| Magma ink | Buy | 8% |
| Ashes of the Phoenix | Sell | 6% |
| Volcanic incense | Buy | 5% |
| Scoria paste | Buy | 4% |
| Obsidian cutlery | Buy | 3% |

This uses 80% of the manual budget and pays 89,200 XIRECS in fees.
The fee rule makes each product's break-even move equal to its
allocation percentage, so the unused 20% is intentional. It avoids
forcing capital into weaker headlines where the quadratic fee can
overwhelm the news edge.